In [1]:
import openeo

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [4]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              112.6669403,
              -7.1613419
            ],
            [
              113.0865997,
              -7.2296572
            ],
            [
              113.0865997,
              -6.8602645
            ],
            [
              112.6725233,
              -6.8602645
            ],
            [
              112.6669403,
              -7.1613419
            ]
          ]
        ]
      }
    }
  ]
}

In [5]:
CO = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent = {
    "west": 112.6669403,
    "south": -7.2296572,
    "east": 113.0865997,
    "north": -6.8602645
    },
    bands=["NO2"],
)

In [6]:
# Now aggregate by day to avoid having multiple data per day
CO = CO.aggregate_temporal_period(reducer="mean", period="day")

# let's create a spatial aggregation to generate mean timeseries data
CO = CO.aggregate_spatial(reducer="mean", geometries=aoi)

In [ ]:
# Create a datacube for period after COVID lockdowns

s5post = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent = {
    "west": 112.6669403,
    "south": -7.2296572,
    "east": 113.0865997,
    "north": -6.8602645
    },
    bands=["NO2"],
)

# Now aggregate by day to avoid having multiple data per day
s5post = s5post.aggregate_temporal_period(reducer="mean", period="day")

# Now create a spatial aggregation to generate mean timeseries data
s5post = s5post.aggregate_spatial(reducer="mean", geometries=aoi)

In [8]:
job = s5post.execute_batch(title="NO2 Bangkalan", outputfile="../data/nc/polutan_NO2_bangkalan.nc")

0:00:00 Job 'j-2608300653274f48ad7d34a381128b03': send 'start'
0:00:03 Job 'j-2608300653274f48ad7d34a381128b03': queued (progress 0%)
0:00:08 Job 'j-2608300653274f48ad7d34a381128b03': queued (progress 0%)
0:00:15 Job 'j-2608300653274f48ad7d34a381128b03': queued (progress 0%)
0:00:23 Job 'j-2608300653274f48ad7d34a381128b03': queued (progress 0%)
0:00:33 Job 'j-2608300653274f48ad7d34a381128b03': queued (progress 0%)
0:00:45 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:01:01 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:01:20 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:01:44 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:02:15 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:02:52 Job 'j-2608300653274f48ad7d34a381128b03': running (progress N/A)
0:03:39 Job 'j-2608300653274f48ad7d34a381128b03': finished (progress 100%)


### Ekspor Data NetCDF (.nc) ke CSV

Sel di bawah ini digunakan untuk memproses file `.nc` hasil crawling di atas, mengekstrak data tanggal (`tanggal`) dan nilai konsentrasi polutan, lalu menyimpannya dalam format `.csv` ke folder `data/csv/`.

In [ ]:
import os
import shutil
import tempfile
import xarray as xr
import pandas as pd

# 1. Konfigurasi polutan
# Anda bisa mengubah nilai ini menjadi "CH4", "CO", "NO2", atau "SO2"
pollutant = "NO2"

# Menangani perbedaan penamaan file (CO menggunakan huruf kecil 'co')
file_pollutant_name = "co" if pollutant.upper() == "CO" else pollutant.upper()
nc_file_path = f"../data/nc/polutan_{file_pollutant_name}_bangkalan.nc"
csv_file_path = f"../data/csv/polutan_{pollutant.lower()}_bangkalan.csv"

print(f"Membaca file: {nc_file_path}")

# Catatan Windows: Jika path folder mengandung karakter non-ASCII (seperti '画像'),
# netCDF4 Windows akan melempar FileNotFoundError. Kita gunakan fallback menyalin file ke folder temp.
is_temp = False
try:
    ds = xr.open_dataset(nc_file_path)
except Exception as e:
    print("Mendeteksi unicode path pada Windows, menggunakan fallback menyalin file ke folder temp...")
    temp_dir = tempfile.gettempdir()
    temp_nc_path = os.path.join(temp_dir, f"temp_{file_pollutant_name}_bangkalan.nc")
    shutil.copy2(nc_file_path, temp_nc_path)
    ds = xr.open_dataset(temp_nc_path)
    is_temp = True

# 2. Konversi dataset NetCDF ke Pandas DataFrame
df = ds.to_dataframe().reset_index()

# 3. Filter dan rename kolom yang dibutuhkan (t -> tanggal, dan nilai polutan)
variable_name = pollutant.upper()

if 't' in df.columns and variable_name in df.columns:
    df_result = df[['t', variable_name]].rename(columns={'t': 'tanggal'})
    
    # 4. Simpan ke file CSV
    os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)
    df_result.to_csv(csv_file_path, index=False)
    
    print(f"\nBerhasil menyimpan ke: {csv_file_path}")
    print("\nContoh data:")
    print(df_result.head())
else:
    print(f"\nError: Kolom 't' atau '{variable_name}' tidak ditemukan di dataset.")
    print("Kolom yang tersedia:", list(df.columns))

# 5. Tutup dataset dan bersihkan file temp jika ada
ds.close()
if is_temp:
    try:
        os.remove(temp_nc_path)
    except Exception:
        pass